# AssistAI
This is assist AI a locally run LLM that has access to your files so it knows who you are

Sources:
- https://github.com/techleadhd/chatgpt-retrieval/blob/main/chatgpt.py
- https://python.langchain.com

## First expirement

I am try and make a call to the locally run model and see if it answers, ideally I want to also give it some context and see if it can answer my questions

In [2]:
import os
import sys

# The overall model
from langchain_community.document_loaders import TextLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.llms import Ollama

llm = Ollama(model="tinyllama")

In [32]:
# Test if llm is loaded
# llm.invoke("how can langsmith help with testing?")

In [33]:
# Ask question based on one document
loader = TextLoader(os.path.join("data", "dogs.md"))
document = loader.load()

prompt = ChatPromptTemplate.from_messages([
    ("system", "Pretend like the only knowledge you have is from the context given"),
    ("user", "{input}")
])

output_parser = StrOutputParser()

chain = prompt | llm | output_parser

chain.invoke({
    "input": "what is the difference between a dog penis and a human penis?",
    "context": document
})

"The human penis is a long, slender penis that grows outside of a man's body. It is typically around 3 to 4 inches (7.6-10 cm) in length, with a diameter ranging from 1.5 to 2 inches (3.8-5.1 cm). On the other hand, the dog penis is a small and short penis that grows inside of a man's body. It typically measures around 1.5 to 2 inches (3.8-5.1 cm) in length. In general, dogs have shorter and more elongated penises than men due to their reproductive system being primarily focused on the female species."

### Conclusions from first experiment

In the first experiment I have found out that one document is not enough to give as context and this is not the correct way. However I have learnt how to use langchain to ask a question to my locally run model and to give it a single document as context

## Second experiment

In the first experiment I saw that it can answer my questions but that one document is by far not enough and that it still uses mostly it's own context. I want to try and give it all files in a folder as context and see how this changes things.

In [3]:
# Getting next level with some extra imports
from langchain_community.document_loaders import DirectoryLoader
from langchain.chains.combine_documents import create_stuff_documents_chain

In [4]:
# Loading all documents
loader = DirectoryLoader('data', loader_cls=TextLoader, glob='**/*.md')
docs = loader.load()

print(len(docs))

138


In [5]:
# Create chain
prompt = ChatPromptTemplate.from_messages([
    ("system", "Find all answers in the documents provided as context \"{context}\""),
    ("user", "{input}")
])

combine_documents_chain = create_stuff_documents_chain(llm, prompt)

In [43]:
# Invoke chat
combine_documents_chain.invoke({
    "input": input("What is your question?"),
    "context": docs
})

'Ubuint is a free, open-source, and cross-platform software suite created by the University of California, Berkeley. It offers various tools for text editing, multimedia creation, scientific computing, and machine learning, among other features. Ubuint is known for its ease of use and versatility, making it an excellent choice for those looking to create innovative projects without sacrificing productivity or performance. With over 3000 contributors and a user base of more than 6 million, Ubuint has become one of the most popular open-source software suites available.'

### Conclusions from second experiment

This already works a lot better sicne there is a lot of context given, however the drawback currently is that my machine takes a long time to answer the question. This would not happen on a machine with a graphics card but this application should be usable for all even without having a GPU.

## Third experiment

In the last experiment there were two problems, the model didnt use the documents as it's main source so hallucinated and it took too long to generate a response.
In the third experiment I want to try and use a different chain called the `RetrievalChain` which first fetches all the corresponding documents before feeding them to the model.

In [6]:
# Import necessary modules
from langchain_community.vectorstores import Chroma
from langchain.chains import create_retrieval_chain
from langchain_community.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings,
)

In [11]:
# Create new chain
embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

PERSIST = True

if os.path.exists("persist") and PERSIST:
    vectorstore = Chroma.from_documents(docs, embedding_function, persist_directory="persist")
elif not os.path.exists("persist") and PERSIST:
    os.makedirs("persist")
    vectorstore = Chroma.from_documents(docs, embedding_function, persist_directory="persist")
else:
    vectorstore = Chroma.from_documents(docs, embedding_function)

retriever = vectorstore.as_retriever(k=10)

retrieval_chain = create_retrieval_chain(retriever, combine_documents_chain)

In [54]:
chat_history = []
query = None

while True:
  if not query:
    query = input("What do you want to know?: ")
  
  if query in ['quit', 'q', 'exit']:
    sys.exit()
  
  result = retrieval_chain.invoke({
      "input": query
  })
  print(result.get('answer', 'I do not know'))

  chat_history.append((query, result['answer']))
  query = None

Go is a modern, powerful, and versatile programming language that has gained widespread popularity in recent years for its simplicity, readability, and ease of use. Here are some of the benefits of using Go:

1. High-level syntax: Go's high-level syntax makes it easy to write concise and efficient code, especially for systems programming or software development.

2. Flexibility: Go supports a wide range of data types and features that make it easier to build scalable applications.

3. Interoperability: The standard library in Go provides an excellent interoperability with other programming languages.

4. Security: Go's secure coding practices, including code review, security testing, and encryption, help to ensure the safety of your applications.

5. Productivity: Go's simple syntax, concise code, and efficient features make it easy to write high-quality software quickly.

6. Compatibility: Go has support for several popular operating systems, including Windows, macOS, Linux, and more,